# 09 — SQL Analytics Sprint

**Project P.U.L.S.E.** — Predictive Unified Life-sciences Summarization Engine

This notebook is a **pure SQL analytics sprint** using DuckDB — structured as the kind of exploratory analysis an Abbott GDSA intern would actually run in the first week of a new project.

**Six analytical questions answered:**
1. Patient demographics by source dataset
2. Diabetes risk stratification into clinical buckets
3. Top biomarker correlates with the diabetes label
4. Longitudinal biomarker trends across NHANES cycles
5. High-risk patient identification for clinical flagging
6. Export high-risk cohort for the clinical team

DuckDB runs entirely in-process with zero server setup, and queries Parquet files directly — no data loading required.

In [ ]:
import duckdb
import pandas as pd

con = duckdb.connect()
con.execute("CREATE VIEW master AS SELECT * FROM read_parquet('data/processed/master_patient_table.parquet')")
print("DuckDB ready. Schema:")
con.execute("DESCRIBE master").df()

## Query 1: Patient Demographics by Dataset

The first query answers the fundamental *who is in our data?* question. Before any modelling, a GDSA analyst needs to understand the **age distribution**, **BMI profile**, and **sex composition** of each contributing dataset.

Demographic differences between datasets are a leading source of **model transfer failure**: a model trained predominantly on a younger population will underperform when deployed on an older one. This query surfaces those differences immediately, informing both the modelling strategy (stratified training vs pooled training) and any required subgroup analyses.

In [ ]:
# Query 1 — Patient demographics by dataset
con.execute("""
    SELECT source_dataset,
           COUNT(*) AS n,
           ROUND(AVG(age), 1) AS avg_age,
           ROUND(STDDEV(age), 1) AS std_age,
           ROUND(AVG(bmi), 1) AS avg_bmi,
           SUM(CASE WHEN gender = 1 THEN 1 ELSE 0 END) AS male_count,
           SUM(CASE WHEN gender = 0 THEN 1 ELSE 0 END) AS female_count
    FROM master
    GROUP BY source_dataset ORDER BY n DESC
""").df()

## Query 2: Diabetes Risk Stratification Buckets

Clinical practice does not think in terms of binary labels — it thinks in terms of **risk tiers**. This query applies the standard ADA (American Diabetes Association) diagnostic thresholds to assign every patient in the master table to one of four categories:

| Category | Glucose | HbA1c |
|---|---|---|
| **Normal** | < 100 mg/dL | < 5.7% |
| **Pre-diabetic** | 100–125 mg/dL | 5.7–6.4% |
| **Diabetic** | ≥ 126 mg/dL | ≥ 6.5% |
| **Insufficient data** | Missing | Missing |

The percentage breakdown across the entire population informs the class balance strategy for modelling and contextualises the clinical relevance of the platform.

In [ ]:
# Query 2 — Diabetes risk stratification buckets
con.execute("""
    SELECT
        CASE
            WHEN glucose < 100 AND hba1c < 5.7 THEN 'Normal'
            WHEN glucose BETWEEN 100 AND 125 OR hba1c BETWEEN 5.7 AND 6.4 THEN 'Pre-diabetic'
            WHEN glucose >= 126 OR hba1c >= 6.5 THEN 'Diabetic'
            ELSE 'Insufficient data'
        END AS risk_category,
        COUNT(*) AS patient_count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) AS pct_of_total
    FROM master
    GROUP BY risk_category ORDER BY patient_count DESC
""").df()

## Query 3: Top Biomarker Correlates with Diabetes Label

Before running SHAP or feature importance from a trained model, a quick Pearson correlation matrix against the target label gives an immediate signal about **which features carry the most predictive information**.

This is not a substitute for model-based feature importance (correlations assume linearity and can mislead in skewed distributions), but it is an invaluable **sanity check**: if HbA1c doesn't show a strong positive correlation with the diabetes label, something is wrong with the label construction or the feature encoding.

Expected ranking: HbA1c > Glucose > BMI > Age > Blood Pressure.

In [ ]:
# Query 3 — Top biomarker correlates with diabetes label
con.execute("""
    SELECT
        ROUND(CORR(glucose, diabetes_label), 3)            AS glucose_corr,
        ROUND(CORR(hba1c, diabetes_label), 3)              AS hba1c_corr,
        ROUND(CORR(bmi, diabetes_label), 3)                AS bmi_corr,
        ROUND(CORR(blood_pressure_sys, diabetes_label), 3) AS bp_corr,
        ROUND(CORR(age, diabetes_label), 3)                AS age_corr
    FROM master WHERE diabetes_label IS NOT NULL
""").df()

## Query 4: NHANES Longitudinal Biomarker Trend

One of P.U.L.S.E.'s core value propositions is its ability to track **population health trends over time** using the two NHANES cycles as a longitudinal panel (2015-16 baseline vs 2021-23 current).

This query computes both the mean and **median** for each biomarker per cycle — a best practice in clinical analytics because median is robust to extreme outliers (e.g., critically ill patients with glucose > 400 mg/dL). Comparing mean and median also reveals skewness: if the mean is substantially above the median, the distribution is right-skewed and the population contains a heavy tail of very high-risk patients.

This analysis directly feeds the Evidently AI drift dashboard in notebook 04.

In [ ]:
# Query 4 — NHANES longitudinal biomarker trend (cross-cycle)
con.execute("""
    SELECT nhanes_cycle,
           ROUND(AVG(glucose), 2)          AS avg_glucose,
           ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY glucose), 2) AS median_glucose,
           ROUND(AVG(hba1c), 2)            AS avg_hba1c,
           ROUND(AVG(bmi), 2)              AS avg_bmi,
           COUNT(*) AS n
    FROM master WHERE source_dataset = 'nhanes' AND nhanes_cycle IS NOT NULL
    GROUP BY nhanes_cycle ORDER BY nhanes_cycle
""").df()

## Query 5: High-Risk Patient Identification

This query implements a **clinical flagging rule** — the SQL equivalent of a clinical decision support alert. It identifies patients who clear the diagnostic thresholds for active diabetes or critical hyperglycaemia and assigns them to a risk tier:

| Tier | Criteria | Clinical interpretation |
|---|---|---|
| **Critical** | Glucose ≥ 200 or HbA1c ≥ 9.0 | Severe uncontrolled diabetes; immediate intervention |
| **High** | Glucose ≥ 126 or HbA1c ≥ 6.5 | Active diabetes diagnosis; management review needed |

In a deployed system (e.g., Abbott's FreeStyle Libre data pipeline), this query would run on incoming sensor data to generate a **priority work list** for clinical care managers, sorted by severity.

In [ ]:
# Query 5 — High risk patient identification (for clinical flagging)
high_risk = con.execute("""
    SELECT patient_id, source_dataset, age, bmi, glucose, hba1c,
           blood_pressure_sys,
           CASE
               WHEN glucose >= 200 OR hba1c >= 9.0 THEN 'Critical'
               WHEN glucose >= 126 OR hba1c >= 6.5 THEN 'High'
               ELSE 'Moderate'
           END AS risk_tier
    FROM master
    WHERE (glucose >= 126 OR hba1c >= 6.5) AND diabetes_label IS NOT NULL
    ORDER BY glucose DESC, hba1c DESC
    LIMIT 20
""").df()
high_risk

## Query 6: Export High-Risk Cohort

The final step exports the high-risk patient list to a CSV for handoff to the clinical operations team. In a production pipeline, this would be an automated step — perhaps a nightly job that writes the latest high-risk cohort to a secure shared drive or EHR integration endpoint.

The CSV format is deliberately simple: it must be readable by clinicians who may not have data tools. Column names use plain English labels, and the `risk_tier` column provides an immediately actionable triage classification without requiring interpretation of raw biomarker values.

**Output file:** `reports/high_risk_patients.csv`

In [ ]:
# Query 6 — Export high-risk patients for clinical team
high_risk.to_csv("reports/high_risk_patients.csv", index=False)
print(f"Exported {len(high_risk)} high-risk patients to reports/high_risk_patients.csv")